In [1]:
### Reading in the parameters from the par files

In [2]:
def norm_cumulative_diff(values, errors, indices, wav, werr):
    """
    Cumulative shifts in the partition mu, normalized by the partition width.
    
    This is taken to mean the difference with respect to first value for the mu in the partition, normalized by (weighted) average sigma taken at the partition level.

    Parameters:
    - parameter1: values is the array of mus for which the difference needs to be calculated.
    - parameter2: errors associated with mus
    - parameter3: indices array marking the length of the partitions
    - parameter4: the (weighted) average sigma for the partition.
    - parameter5: error associated with the weighted average sigma at the partition-level

    Returns:
    Array contains elements given by mu_ij - mu_0j for every partition j
    """
    
    # array with the differences and errors is initialized
    norm_cum_diff     = []
    norm_cum_diff_unc = []
        
    
    for idx in range(len(indices)-1):
        # indices were shifted by 0.5 for visualization purposes, so shifted back for referencing purposes
        start_idx = int(indices[idx] + 0.5)
        end_idx   = int(indices[idx + 1] + 0.5)
        
        # Slice the values and errors for the current segment
        segment_values = values[start_idx:end_idx]
        segment_errors = errors[start_idx:end_idx]
        
        # If the first element is nan, set the reference to 0
        if not np.isnan(segment_values[0]):
            ref_el  = segment_values[0]
            ref_err = segment_errors[0]
        else:
            ref_el  = 0
            ref_err = 0

        # Calculate the differences and errors
        for i, el in enumerate(segment_values):
            cum_diff_value      = (el - ref_el)
            norm_cum_diff_value = cum_diff_value/wav[idx]

            cum_diff_unc_value      = np.sqrt(segment_errors[i]**2 + ref_err**2)
            norm_cum_diff_unc_value = abs(norm_cum_diff_value) * np.sqrt ((cum_diff_unc_value/cum_diff_value)**2 + (werr[idx]/wav[idx])**2)
            
            norm_cum_diff.append(norm_cum_diff_value)
            norm_cum_diff_unc.append(norm_cum_diff_unc_value)
            
    return norm_cum_diff, norm_cum_diff_unc

In [3]:
def mod_diff(arr, err):
    """
    This is similar to the numpy.diff function but there is a small difference:
    If the previous point is NaN, the code will search for a non-NaN previous point to do the subtraction with.

    Parameters:
    - parameter1: arr is the array for which the difference between elements needs to be calculated.
    - parameter2: err is the array of errors associated with the elements

    Returns:
    - The array whose elements are the absolute difference between the elements of the original array and the associated errors.
    - The associated uncertainties.
    
    Note that the differences should be absolute values!
    """
    
    # if the first element is NaN, set previous element to 0, else set use it
    # set the error correspondingly
    if not np.isnan(arr[0]):
        prev_el  = arr[0]
        prev_err = err[0]
    else:
        prev_el  = 0
        prev_err = 0
        
    # array with the differences and errors is initialized
    diff     = []
    diff_unc = []
    
    # Calculate the differences and errors
    for i, curr_el in enumerate(arr):
        jk = i
        diff.append(abs(curr_el - prev_el))
        diff_unc.append(np.sqrt(err[i]**2 + prev_err**2))
        
        # Setting the next instance of the previous element
        if not np.isnan(curr_el):
            prev_el  = curr_el
            prev_err = err[i]
        else:
            while np.isnan(arr[jk]):
                jk      -= 1
                prev_el  = arr[jk]
                prev_err = err[jk]
                
    return diff, diff_unc

In [4]:
def weighted_average_and_uncertainty(values, errors, indices):
    """
    Calculate the weighted average and its uncertainty when weights are the
    inverse of the uncertainty squared.

    Parameters:
    - values: NumPy array of values.
    - uncertainties: NumPy array of uncertainties corresponding to the values.

    Returns:
    - weighted_average
    """
    weighted_av     = []
    
    for idx in range(len(indices)-1):
        start_idx = int(indices[idx] + 0.5)
        end_idx   = int(indices[idx + 1] + 0.5)
        
        # Slice the values and errors for the current segment
        segment_values = np.asarray(values[start_idx:end_idx], dtype=float)
        segment_errors = np.asarray(errors[start_idx:end_idx], dtype=float)
        
        if len(segment_values) != len(segment_errors):
            raise ValueError("The length of values and uncertainties must be the same.")

        # Calculate weights as the inverse of the uncertainty
        weights = 1 / (segment_errors)
        
        try:
            mask = ~np.isnan(segment_values)
            # Calculate weighted average
            weighted_av_value = np.average(segment_values[mask], weights=weights[mask])
    
            # Calculate uncertainty of the weighted average
            total_weight = np.sum(weights[mask])
            
        except:
            weighted_av_value = np.nan
        
        weighted_av.append(weighted_av_value)
    
    return weighted_av

In [5]:
def index_marker(diff, sigma,pd_start):
    """
    This function returns the indices for which the diff is greater than sigma/4.

    Parameters:
    - parameter1: diff is the array of differences. The first element is either 0 or nan by default.
                  Thus, the number of indices is equal to the number of file_tags.
                  This makes the assumption that at least the first file will have to lie within the partition.
    - parameter2: sigma is the array of 1 sigma.
    - parameter3: List of the start of all periods (except the first one which is already taken into account)
    
    The comparison is ith element of diff with the ith element of the sigma array.
    To take an example, diff[2] = mu[2] - mu[1] which is compared to sigma[2].

    Returns:
    The ind array which contains the points where we need to draw new partitions.
    """
    thr = np.asarray(sigma, dtype = float)/4
    temp_mark = np.where(diff > thr)
    indices   = np.concatenate(([0], temp_mark[0], pd_start, [len(diff)]))
    # Sort the array
    indices.sort()

    # Drop duplicates
    indices = np.unique(indices)
    
    colors  = []
    for i in range(len(indices) - 1):
        if   (i%3 == 0): 
            colors.append('grey')
        elif (i%3 == 1):
            colors.append('orange')
        elif (i%3 == 2):
            colors.append('blue')
            
    indices = [x-0.5 for x in indices]

    return indices, colors

In [6]:
def pull_datetimestamp(calfile):
    try:
        start_ind = calfile.index("-cal-") + len("-cal-")
        stop_ind = calfile.index("-tier")
    except:
        start_ind = calfile.index("-phy-") + len("-phy-")
        stop_ind = calfile.index("-par")
    dt = calfile[start_ind:stop_ind]
    return dt

In [7]:
# def check_validity(channel_name, cuts_string, low_aoe, high_aoe, lq, ann, coax_rt):
#     cuts_map = {'low_aoe':low_aoe, 'high_aoe':high_aoe, 'lq':lq, 'ann':ann, 'coax_rt':coax_rt}
#     # Sometimes the cuts_string is just 'missing' so this case has been taken into consideration.
#     if cuts_string == 'missing':
#         print(f"The format is different for {channel_name} and the cut string is missing.")
#         return False
#     # If not the above, proceed to split the string into the various cuts
#     cuts = cuts_string.split('&')  # Split the string into individual cuts
#     for cut in cuts:
#         cut = cut.strip()  # Remove any leading/trailing whitespace
#         # Go back and check whether the cut is valid or not.
#         #print(cuts_map[cut])
#         if cuts_map[cut]!= 'valid':
#             return False  # Return False if any cut is not valid
#     return True  # Return True if all cuts are valid

In [8]:
def pull_chmap(dt):
    # dbetto change means that the if () statements need to be changed into the hasattr commands
    chmap = lmeta.hardware.configuration.channelmaps.on(dt)
    
    channel_dict_method = {}
    
    for channel_name, channel_data in chmap.items():     
        if channel_data.system == "geds":
            #print(channel_name)
            # Pull out the analysis flags based on the cal status
            ged = lmeta.channelmap(dt)[channel_name]
            ch=f'ch{ged["daq"]["rawid"]}'
            ged_ana = lmeta.datasets.statuses.on(dt, system="cal")[channel_name]

            channel_dict_method[ch] = (lmeta.channelmap(dt)[channel_name].type,
                                channel_name,
                                lmeta.channelmap(dt)[channel_name].production['mass_in_g'],
                                lmeta.channelmap(dt)[channel_name].location.string,
                                lmeta.channelmap(dt)[channel_name].location.position)
    return channel_dict_method

In [9]:
# Importing the packages
import json # to read/write the json files
import numpy as np # for numpy arrays
import pandas as pd # for pandas dataframes

from legendmeta import LegendMetadata # for querying the detector database
from datetime import datetime # for pulling channel maps relating to a specific time

import os
import glob

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import ticker

from functools import reduce

from matplotlib.patches import Rectangle

import re

import warnings
warnings.filterwarnings("ignore") # fed up of the load_dfs warnings

In [10]:
# Path to legend-metadata
# If pointing to a user specific path, remember to put it inside the paranthesis
lmeta = LegendMetadata()

could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f6c7d1dc860>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f6c7d1dc770>)
could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f6c7d1ddd00>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f6c7d1dd760>)
could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f6c7d018270>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f6c7d018bd0>)
could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <

In [11]:
file_tags = [#"p15_r002", "p15_r003", "p15_r004", "p15_r005", 
             "p16_r000", "p16_r001", "p16_r002", "p16_r003", "p16_r004", "p16_r005", "p16_r006", "p16_r007",
             "p18_r000", "p18_r001", "p18_r002", "p18_r003", "p18_r004", "p18_r005", "p18_r006",#,
             "p19_r001", "p19_r002", "p19_r003", "p19_r005", "p19_r006"]#, "p19_r007", "p19_r008", "p19_r009"]

In [12]:
# # Set the directory path
# directory_path = '/pscratch/sd/j/jita/jita/pygama-tut/AoE_systematics/runwise-stability-v2/fit-files/v6-cclow/'
# files          = sorted(glob.glob(f"{directory_path}*.txt"))

# file_tags   = []

# # Initialize an empty list to store DataFrames
# data_frames = []

# # Iterate through files in the directory
# for file in files:
#     if (file[file.index('l200')+5:file.index('.txt')])!= 'p07-r001-cal':
#         file_tags.append(file[file.index('l200')+5:file.index('.txt')])

#         # Load each file into a DataFrame
#         df = pd.read_csv(file, sep='\t')

#         # Append the DataFrame to the list
#         data_frames.append(df)

#     # Set the index as the key and then concatenate
#     data_frames_with_index = [df.set_index('Key') for df in data_frames]
#     result_df = pd.concat(data_frames_with_index, axis=1)

In [13]:
# # Select relevant columns
# mu_col     = [x for x in range(0, result_df.shape[1], 4)]  
# muerr_col  = [x for x in range(1, result_df.shape[1], 4)]  
# sig_col    = [x for x in range(2, result_df.shape[1], 4)] 
# sigerr_col = [x for x in range(3, result_df.shape[1], 4)]  

# # Split the DataFrame into different parts based on columns
# df_mu      = result_df.iloc[:, mu_col]  
# df_muerr   = result_df.iloc[:, muerr_col]  
# df_sig     = result_df.iloc[:, sig_col]  
# df_sigerr  = result_df.iloc[:, sigerr_col] 

In [14]:
# Using the datetimestamp of March 2023
chmap = lmeta.hardware.configuration.channelmaps.on('20250911T235840Z')
channel_dict = {}
for channel_name, channel_data in chmap.items():
    if (channel_data.system == "geds"):
        channel_dict[channel_data["daq"]["rawid"]] = (channel_name,
                                                      channel_data.location.string,
                                                      channel_data.location.position,
                                                      channel_data.location.string*100 + channel_data.location.position)

In [15]:
detlist = [channel_dict[key][0] for key in channel_dict.keys()]
# channel df
channel_df = pd.DataFrame.from_dict(channel_dict, orient='index',\
                                    columns=['channel_name',
                                             'string',
                                             'posn',
                                             'short_mageid'])

In [16]:
pardict = {'01':{'p16':['r000', 'r001', 'r002', 'r003', 'r004', 'r005', 'r006', 'r007']},
           '02': {'p18': ['r000', 'r001', 'r002', 'r003', 'r004', 'r005', 'r006']},
           '03': {'p19': ['r001', 'r002', 'r003', 'r005', 'r006']}}
pd_start = [8,15,20] #[8,15,25]

In [17]:
import yaml
bl_pars = {}
df_mu = {}
df_muerr = {}
df_sig = {}
df_sigerr = {}

for file_tag in file_tags:
    bl_pars[file_tag] = {}
    df_mu[file_tag]= {}
    df_muerr[file_tag] = {}
    df_sig[file_tag] = {}
    df_sigerr[file_tag] = {}
    basedir  = '/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/tmp/v3.1.0dev5/generated'
    parfile  = glob.glob(f'{basedir}/par/hit/cal/{file_tag[0:3]}/{file_tag[4:8]}/*-par_hit.yaml')[0]

    fo   = open(parfile) # io object to open the file
    data = yaml.safe_load(fo) # load json file
    for ch in channel_dict.keys():
        bl_pars[file_tag][f'ch{ch}']= {}
        df_mu[file_tag][f'ch{ch}'] = {}
        df_muerr[file_tag][f'ch{ch}'] = {}
        df_sig[file_tag][f'ch{ch}'] = {}
        df_sigerr[file_tag][f'ch{ch}'] = {}
        try:
            ts = list(data[channel_dict[ch][0]]['results']['aoe']['1000-1300keV'].keys())[0]
            print(ch, file_tag, ts)
            bl_pars[file_tag][f'ch{ch}']['bl_std']       = data[channel_dict[ch][0]]['results']['ecal']['monitoring_parameters']['bl_std']['mode']
            bl_pars[file_tag][f'ch{ch}']['baselineEmax'] = data[channel_dict[ch][0]]['results']['ecal']['monitoring_parameters']['baselineEmax']['mode']
            df_mu[file_tag][f'ch{ch}'] = data[channel_dict[ch][0]]['results']['aoe']['1000-1300keV'][ts]['mean']
            df_muerr[file_tag][f'ch{ch}'] = data[channel_dict[ch][0]]['results']['aoe']['1000-1300keV'][ts]['mean_err']
            df_sig[file_tag][f'ch{ch}'] = data[channel_dict[ch][0]]['results']['aoe']['1000-1300keV'][ts]['sigma']
            df_sigerr[file_tag][f'ch{ch}'] = data[channel_dict[ch][0]]['results']['aoe']['1000-1300keV'][ts]['sigma_err']
        except:
            bl_pars[file_tag][f'ch{ch}']['bl_std']       = np.nan
            bl_pars[file_tag][f'ch{ch}']['baselineEmax'] = np.nan
            df_mu[file_tag][f'ch{ch}'] = np.nan
            df_muerr[file_tag][f'ch{ch}'] = np.nan
            df_sig[file_tag][f'ch{ch}'] = np.nan
            df_sigerr[file_tag][f'ch{ch}'] = np.nan
            continue

1104004 p16_r000 20000101T000000Z
1104005 p16_r000 20000101T000000Z
1105600 p16_r000 20000101T000000Z
1107200 p16_r000 20000101T000000Z
1107201 p16_r000 20000101T000000Z
1107202 p16_r000 20000101T000000Z
1107203 p16_r000 20000101T000000Z
1107205 p16_r000 20000101T000000Z
1108800 p16_r000 20000101T000000Z
1108801 p16_r000 20000101T000000Z
1108802 p16_r000 20000101T000000Z
1105602 p16_r000 20000101T000000Z
1108803 p16_r000 20000101T000000Z
1108804 p16_r000 20000101T000000Z
1108805 p16_r000 20000101T000000Z
1110400 p16_r000 20000101T000000Z
1110401 p16_r000 20000101T000000Z
1110402 p16_r000 20000101T000000Z
1110403 p16_r000 20000101T000000Z
1110404 p16_r000 20000101T000000Z
1110405 p16_r000 20000101T000000Z
1112000 p16_r000 20000101T000000Z
1112001 p16_r000 20000101T000000Z
1112002 p16_r000 20000101T000000Z
1112003 p16_r000 20000101T000000Z
1112004 p16_r000 20000101T000000Z
1113600 p16_r000 20000101T000000Z
1113603 p16_r000 20000101T000000Z
1113604 p16_r000 20000101T000000Z
1113605 p16_r0

In [18]:
pdf_pages = PdfPages("Partition_p16_p18_p19.pdf")

det_names = [channel_dict[ch][0] for ch in channel_dict.keys()]

for i, rawid in enumerate(channel_dict.keys()):
    print(f"Detector:{channel_dict[rawid][0]}\u2714")
    # Create figure and subplots
    fig, axs = plt.subplots(nrows=3, ncols=2, sharex=True, figsize=(15, 8))
    fig.suptitle(f"RawID: {rawid}, Detector: {channel_dict[rawid][0]}, String: {channel_dict[rawid][1]}, Posn:{channel_dict[rawid][2]}")
    try:
        diff, diff_unc  = mod_diff([df_mu[tag][f'ch{rawid}'] for tag in file_tags], [df_muerr[tag][f'ch{rawid}'] for tag in file_tags])
        indices, colors = index_marker(diff, [df_sig[tag][f'ch{rawid}'] for tag in file_tags], pd_start)
                
        axs[0,0].errorbar(file_tags,
                        y         = [df_mu[tag][f'ch{rawid}'] for tag in file_tags],
                        yerr      = [df_muerr[tag][f'ch{rawid}'] for tag in file_tags],
                        marker    ='o',
                        linestyle ='--',
                        label     =r'$\mu$',
                        color     ='blue')
        axs[0,0].set_ylabel(r'$\mu$ [ADC]', color='blue')
        axs[0,0].grid(linestyle='dashed', linewidth=0.5)
        # Shade in partitions
        for j, element in enumerate(indices[:-1]):
            axs[0,0].axvspan(indices[j], indices[j+1], alpha=0.2, color=colors[j])
        axs[0,0].legend(loc='upper left')
        axs[0,0].tick_params(labelbottom=False)
    
        axs[0,1].errorbar(file_tags,
                      y         = [df_sig[tag][f'ch{rawid}'] for tag in file_tags],
                      yerr      = [df_sigerr[tag][f'ch{rawid}'] for tag in file_tags],
                      marker    ='s',
                      linestyle ='--',
                      label     =r'$\sigma$',
                      color     ='red')
        axs[0,1].set_ylabel(r'$\sigma$ [ADC]', color='red')
        axs[0,1].grid(linestyle='dashed', linewidth=0.5)
        # Shade in partitions
        for j, element in enumerate(indices[:-1]):
            axs[0,1].axvspan(indices[j], indices[j+1], alpha=0.2, color=colors[j])
        axs[0,1].legend(loc='upper right')
        axs[0,1].tick_params(labelbottom=False)
        
        norm_mudiff     = [x/y for x,y in zip(diff, [df_sig[tag][f'ch{rawid}'] for tag in file_tags])]
        norm_mudiff_err = [a/c * np.sqrt((b/a)**2 + (d/c)**2) for a,b,c,d in zip(diff, diff_unc, [df_sig[tag][f'ch{rawid}'] for tag in file_tags], [df_sigerr[tag][f'ch{rawid}'] for tag in file_tags])]
    
        wav = weighted_average_and_uncertainty([df_sig[tag][f'ch{rawid}'] for tag in file_tags], [df_sigerr[tag][f'ch{rawid}'] for tag in file_tags], indices)
        # Error is not taken as the error on the mean but instead the average of the errors...
        werr = []
        for idx in range(len(indices)-1):
            start_idx = int(indices[idx] + 0.5)
            end_idx   = int(indices[idx + 1] + 0.5)
            werr.append(np.nanmean([df_sigerr[tag][f'ch{rawid}'] for tag in file_tags][start_idx:end_idx]))
        norm_cum_diff, norm_cum_diff_unc = norm_cumulative_diff([df_mu[tag][f'ch{rawid}'] for tag in file_tags], [df_muerr[tag][f'ch{rawid}'] for tag in file_tags], indices, wav, werr)
       
        for j in range(len(indices)-1):
            axs[0,1].hlines(wav[j], indices[j], indices[j+1], color = 'black')
            axs[0,1].fill_between([indices[j], indices[j+1]],
                                  wav[j] - 3*werr[j],
                                  wav[j] + 3*werr[j], alpha = 0.3, color = colors[j])
            axs[0,1].hlines(wav[j] - 3*werr[j], indices[j], indices[j+1], color = 'black', linestyle = 'dotted')
            axs[0,1].hlines(wav[j] + 3*werr[j], indices[j], indices[j+1], color = 'black', linestyle = 'dotted')
            
        axs[1,0].errorbar(file_tags,
                    norm_mudiff,
                    norm_mudiff_err,
                    marker    ='o',
                    linestyle ='--',
                    label     ='sudden shifts',
                    color     ='green')
        axs[1,0].set_ylabel(r'|$\frac{\mu_{i+1}-\mu_{i}}{\sigma_{i}}|$')
        axs[1,0].axhline(0.25, linestyle ='--', color = 'red')
        axs[1,0].set_ylim(0, 0.5)
        axs[1,0].grid(linestyle ='dashed', linewidth=0.5)
        # Shade in partitions
        for j, element in enumerate(indices[:-1]):
            axs[1,0].axvspan(indices[j], indices[j+1], alpha=0.2, color=colors[j])
        axs[1,0].legend(loc='upper left')
        axs[1,0].tick_params(labelbottom=False)
    
        axs[1,1].plot(file_tags,
                         [bl_pars[tag][f'ch{rawid}']['bl_std'] for tag in file_tags],
                         marker    ='o',
                         linestyle ='--',
                         label     ='bl_std',
                         color     ='blue')
        axs[1,1].tick_params(axis='x', labelrotation=90)
        axs[1,1].grid(linestyle='dashed', linewidth=0.5)
        axs[1,1].set_ylabel('bl_std')
        # Shade in partitions
        for j, element in enumerate(indices[:-1]):
            axs[1,1].axvspan(indices[j], indices[j+1], alpha=0.2, color=colors[j])
        axs[1,1].legend(loc='upper right')
        axs[1,1].tick_params(labelbottom=False)
        
        axs[2,0].errorbar(file_tags,
                    norm_cum_diff,
                    norm_cum_diff_unc,  
                    marker    ='o',
                    linestyle ='--',
                    label     ='slow shifts',
                    color     ='green')
        axs[2,0].set_xlabel('Run')
        axs[2,0].set_ylabel(r'$\frac{\mu_{i}-\mu_{0,part_{j}}}{\sigma_{part_{j}}}$')
        axs[2,0].axhline(0.5, linestyle ='--', color = 'red')
        axs[2,0].axhline(-0.5, linestyle='--', color = 'red')
        axs[2,0].set_ylim(-1, 1)
        axs[2,0].set_xlabel('Run')
        axs[2,0].tick_params(axis='x', labelrotation=90)
        axs[2,0].grid(linestyle='dashed', linewidth=0.5)
        # Shade in partitions
        for j, element in enumerate(indices[:-1]):
            axs[2,0].axvspan(indices[j], indices[j+1], alpha=0.2, color=colors[j])
        axs[2,0].set_xlim(-1, len(file_tags))
        axs[2,0].legend(loc='upper left')
    
        axs[2,1].plot(file_tags,
                         [bl_pars[tag][f'ch{rawid}']['baselineEmax'] for tag in file_tags],
                         marker    ='o',
                         linestyle ='--',
                         label     ='baselineEmax',
                         color     ='blue')
        axs[2,1].set_xlabel('Run')
        axs[2,1].set_ylabel('baselineEmax')
        axs[2,1].tick_params(axis='x', labelrotation=90)
        axs[2,1].grid(linestyle='dashed', linewidth=0.5)
        # Shade in partitions
        for j, element in enumerate(indices[:-1]):
            axs[2,1].axvspan(indices[j], indices[j+1], alpha=0.2, color=colors[j])
        axs[2,1].legend(loc='upper right')
        plt.subplots_adjust(hspace=0)  # Set space between rows to zero
        # Adjust layout and save the figure
        plt.tight_layout()
        pdf_pages.savefig(fig)
        plt.close(fig)

    except:
        continue

pdf_pages.close()

Detector:V14654A✔
Detector:V14673A✔
Detector:V13044A✔
Detector:B00032C✔
Detector:V07302A✔
Detector:V00050A✔
Detector:V05267A✔
Detector:V00048A✔
Detector:V00048B✔
Detector:V01240A✔
Detector:V05268A✔
Detector:V05261A✔
Detector:V03421A✔
Detector:V03422A✔
Detector:V10447B✔
Detector:V08682A✔
Detector:V09724A✔
Detector:V11924A✔
Detector:V09374A✔
Detector:V02160A✔
Detector:V02162B✔
Detector:V05612A✔
Detector:V05266A✔
Detector:V07647A✔
Detector:V07647B✔
Detector:V07302B✔
Detector:V05266B✔
Detector:V14618A✔
Detector:V13049A✔
Detector:V13046A✔
Detector:B00000C✔
Detector:B00061B✔
Detector:B00076C✔
Detector:B00079B✔
Detector:B00079C✔
Detector:V06649M✔
Detector:V01404A✔
Detector:B00000D✔
Detector:B00002C✔
Detector:B00035A✔
Detector:B00035B✔
Detector:V00050B✔
Detector:V04549A✔
Detector:V07298B✔
Detector:V06643A✔
Detector:V00074A✔
Detector:V08682B✔
Detector:V11925A✔
Detector:V11947B✔
Detector:V09372A✔
Detector:V10784A✔
Detector:V10437B✔
Detector:V02160B✔
Detector:V04199A✔
Detector:V04545A✔
Detector:V